In [2]:
import nltk
import sklearn_crfsuite
from sklearn_crfsuite import metrics

# 1. Definizione delle Feature
# Un CRF ha bisogno di feature per ogni parola (es. la parola stessa, se è maiuscola, suffissi, ecc.)
# e informazioni sul contesto (parola precedente/successiva).
def word2features(sent, i):
    word = sent[i][0]
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],  # Suffix
        'word[-2:]': word[-2:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
    }
    if i > 0:
        word1 = sent[i-1][0]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
        })
    else:
        features['BOS'] = True  # Beginning of Sentence

    if i < len(sent)-1:
        word1 = sent[i+1][0]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
        })
    else:
        features['EOS'] = True  # End of Sentence

    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, label in sent]

# 2. Caricamento Dati di Esempio (utilizzando NLTK Treebank)
nltk.download('treebank')
corpus = nltk.corpus.treebank.tagged_sents()

# Dividiamo in Train e Test set
train_sents = corpus[:3000]
test_sents = corpus[3000:]

X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]

X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]

# 3. Addestramento del Modello CRF
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,  # Regolarizzazione L1
    c2=0.1,  # Regolarizzazione L2
    max_iterations=100,
    all_possible_transitions=True
)
crf.fit(X_train, y_train)

# 4. Valutazione
y_pred = crf.predict(X_test)
labels = list(crf.classes_)
labels.remove('-NONE-') if '-NONE-' in labels else None # Rimuoviamo tag non informativi se presenti

sorted_labels = sorted(labels, key=lambda name: (name[1:], name[0]))
print(metrics.flat_classification_report(y_test, y_pred, labels=sorted_labels, digits=3, zero_division=0))

from collections import Counter

# Raccoglie il conteggio dei tag reali e predetti
true_tags = Counter(tag for sent in y_test for tag in sent)
pred_tags = Counter(tag for sent in y_pred for tag in sent)

# Stampa i risultati allineati in colonne
for tag in sorted(crf.classes_):
    print(f"{tag:<10} true={true_tags[tag]:<5d} pred={pred_tags[tag]:<5d}")





print("First sentence:\n")

for (token, gt), pred in zip(test_sents[0], y_pred[0]):
    print(f"{token:<15} GT={gt:<10} PRED={pred}")




[nltk_data] Downloading package treebank to
[nltk_data]     /home/simonecaputo/nltk_data...
[nltk_data]   Package treebank is already up-to-date!


              precision    recall  f1-score   support

           #      1.000     1.000     1.000         3
           $      1.000     1.000     1.000       255
           ,      1.000     1.000     1.000      1106
           .      1.000     1.000     1.000       891
           :      1.000     1.000     1.000        81
          ''      1.000     1.000     1.000        92
          RB      0.909     0.894     0.901       545
          VB      0.940     0.940     0.940       550
         VBD      0.955     0.946     0.951       904
         VBG      0.921     0.908     0.914       306
         VBN      0.906     0.851     0.877       522
         VBP      0.855     0.898     0.876       177
         RBR      0.571     0.444     0.500        18
         RBS      0.857     1.000     0.923         6
         VBZ      0.957     0.925     0.940       358
          CC      0.996     0.998     0.997       503
          CD      0.995     0.934     0.963      1208
          MD      0.996    

/home/simonecaputo/miniforge3/envs/my_project/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/simonecaputo/miniforge3/envs/my_project/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/simonecaputo/miniforge3/envs/my_project/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_pr

NameError: name 'report' is not defined